# bible-atlas-agent demo

LangChain + LangGraph를 사용한 간단한 데모입니다.

실행 전 `.env` 파일에 `OPENAI_API_KEY` 값을 채워주세요.

## 1. 환경변수 로드 & LLM 생성

In [1]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

llm.model_name

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 2. AgentState 정의

In [ ]:
class AgentState(TypedDict):
    context: list
    query: str
    answer: str

## 3. 노드 정의

`generate_answer` 노드는 `query`와 `context`를 프롬프트에 넣어 LLM을 호출한 뒤, 변경된 필드(`answer`)만 반환합니다.

In [ ]:
def generate_answer(state: AgentState) -> dict:
    context = state["context"]
    query = state["query"]

    response = llm.invoke(
        f"""
        다음 Context를 참고해 질문에 답변하세요.

        Context:
        {context}

        Question:
        {query}
        """
    )

    return {"answer": response.content}

## 4. Graph 구성 & 컴파일

```text
START → generate_answer → END
```

In [ ]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("generate_answer", generate_answer)
graph_builder.add_edge(START, "generate_answer")
graph_builder.add_edge("generate_answer", END)

graph = graph_builder.compile()
graph

### (선택) Graph 시각화

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 5. 초기 상태로 Graph 실행

In [ ]:
initial_state: AgentState = {
    "context": [],
    "query": "아가야는 어떤 곳이야?",
    "answer": "",
}

final_state = graph.invoke(initial_state)
final_state

## 6. 최종 answer 출력

In [ ]:
print(final_state["answer"])